In [1]:
import torch
import torch.nn as nn
import numpy as np
import os

from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu


In [4]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [5]:
dataset = ImageFolder(
    "../dataset/EuroSAT",
    transform=transform
)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

print("Total Images:", len(dataset))

Total Images: 27000


In [6]:
model = models.resnet18(weights=None)

model.fc = nn.Linear(
    model.fc.in_features,
    10
)

model.load_state_dict(
    torch.load("../models/resnet18_best.pth")
)

model = model.to(device)

model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [7]:
feature_extractor = nn.Sequential(
    *list(model.children())[:-1]
)

feature_extractor = feature_extractor.to(device)

feature_extractor.eval()

Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (

In [8]:
embeddings = []

labels = []

with torch.no_grad():

    for images, target in loader:

        images = images.to(device)

        features = feature_extractor(images)

        features = features.view(
            features.size(0),
            -1
        )

        embeddings.append(
            features.cpu().numpy()
        )

        labels.extend(target.numpy())

In [9]:
embeddings = np.concatenate(
    embeddings,
    axis=0
)

labels = np.array(labels)

print("Embedding Shape:", embeddings.shape)

Embedding Shape: (27000, 512)


In [10]:
os.makedirs("../outputs", exist_ok=True)

np.save(
    "../outputs/embeddings.npy",
    embeddings
)

np.save(
    "../outputs/labels.npy",
    labels
)

print("Embeddings Saved Successfully!")

Embeddings Saved Successfully!


In [11]:
embeddings = np.load("../outputs/embeddings.npy")

labels = np.load("../outputs/labels.npy")

print("Embeddings Shape:", embeddings.shape)
print("Labels Shape:", labels.shape)

Embeddings Shape: (27000, 512)
Labels Shape: (27000,)


In [12]:
print("First embedding vector (first 10 values):")
print(embeddings[0][:10])

print("\nTotal embeddings:", embeddings.shape[0])
print("Embedding dimension:", embeddings.shape[1])

First embedding vector (first 10 values):
[0.18263802 1.6746726  2.0953345  0.6320566  0.728508   1.0779332
 0.05784785 0.33306524 0.41825742 0.2686239 ]

Total embeddings: 27000
Embedding dimension: 512
